# Assignment-8

**UCS420 “Cognitive Computing”**

**NLP-II**

**Adarsh Sharma**

**Roll No: 1024170359**

**Batch: 3Q33**

Cognitive Computing Assignment Submission

This notebook answers all six questions in Lab Assignment 10: text preprocessing, stemming and lemmatization, Bag of Words and TF-IDF, similarity measures, sentiment analysis, word clouds, and text generation.

## Q.1 Text preprocessing and word frequency

The paragraph below is about gaming technology. The code converts it to lowercase, removes punctuation, tokenizes words and sentences, compares Python's `split()` with NLTK's `word_tokenize()`, removes stopwords, and displays the frequency distribution of the remaining words.

In [ ]:
import re
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

paragraph = (
    "Gaming technology has changed the way people play and compete. "
    "Modern games use realistic graphics, intelligent characters, and detailed virtual worlds. "
    "Online multiplayer modes allow friends to cooperate even when they live far apart. "
    "Fast processors and powerful graphics cards help games run smoothly. "
    "Cloud gaming can make high-quality games available on more devices. "
    "I enjoy gaming because it combines creativity, strategy, teamwork, and technology."
)

print("Original paragraph:\n", paragraph)
lower_text = paragraph.lower()
clean_text = re.sub(r"[^a-z0-9\s]", "", lower_text)
print("\nLowercase text with punctuation removed:\n", clean_text)

word_tokens = word_tokenize(lower_text)
sentence_tokens = sent_tokenize(paragraph)
print("\nWord tokens:", word_tokens)
print("\nSentence tokens:")
for i, sentence in enumerate(sentence_tokens, 1):
    print(f"{i}. {sentence}")

split_tokens = clean_text.split()
nltk_tokens = [w for w in word_tokenize(clean_text) if w.isalnum()]
print("\nPython split() tokens:", split_tokens)
print("\nNLTK word_tokenize() tokens:", nltk_tokens)
print("\nTokens only in split():", sorted(set(split_tokens) - set(nltk_tokens)))
print("Tokens only in word_tokenize():", sorted(set(nltk_tokens) - set(split_tokens)))

stop_words = set(stopwords.words("english"))
filtered_tokens = [word for word in nltk_tokens if word not in stop_words]
print("\nTokens after stopword removal:", filtered_tokens)

frequency = Counter(filtered_tokens)
frequency_df = pd.DataFrame(frequency.most_common(), columns=["Word", "Frequency"])
display(frequency_df)
frequency_df.plot(kind="bar", x="Word", y="Frequency", legend=False, figsize=(10, 4), title="Word Frequency (excluding stopwords)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### Observation

`split()` divides a string wherever whitespace occurs. `word_tokenize()` uses language-aware tokenization rules and can separate punctuation and contractions into individual tokens. In this example punctuation is removed before the comparison, so the two lists are expected to be very similar. Stopword removal reduces common words such as “the”, “and”, and “is”, making the more informative terms easier to inspect.

## Q.2 Regex extraction, stemming and lemmatization

The same paragraph is used. `re.findall()` extracts alphabetic words, NLTK stopwords are removed, and the remaining words are transformed with `PorterStemmer` and `WordNetLemmatizer`.

In [ ]:
alphabetic_words = re.findall(r"\b[A-Za-z]+\b", paragraph.lower())
content_words = [word for word in alphabetic_words if word not in stop_words]

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemming_rows = []
for word in content_words:
    stemming_rows.append({
        "Original word": word,
        "Porter stem": porter.stem(word),
        "Lemma (noun)": lemmatizer.lemmatize(word),
        "Lemma (verb)": lemmatizer.lemmatize(word, pos=wordnet.VERB)
    })

print("Alphabetic words:", alphabetic_words)
print("\nWords after stopword removal:", content_words)
display(pd.DataFrame(stemming_rows))

### Stemming vs. lemmatization

- **Stemming** removes or alters word endings using heuristic rules. It is usually fast, but the result may not be a valid English word (for example, `creativity` may be shortened to a stem).
- **Lemmatization** uses vocabulary and grammatical information to return a meaningful base form. Its result depends on the part of speech supplied; for example, the verb form `changed` can be lemmatized to `change`.
- I would prefer **stemming** for quick information retrieval where grouping related word forms is more important than readability. I would prefer **lemmatization** for user-facing text analysis or sentiment analysis where meaningful words are useful.

## Q.3 Bag of Words and TF-IDF

Three original short texts about technology are vectorized. `CountVectorizer` produces raw word counts, while `TfidfVectorizer` gives more weight to terms that are informative within a text and less weight to terms shared across the collection. The top three terms for each text are printed.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "Artificial intelligence helps computers learn patterns and make useful predictions.",
    "Blockchain stores digital transactions in a shared and secure distributed ledger.",
    "Cloud computing provides scalable storage and computing resources over the internet."
]

count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vectorizer.get_feature_names_out(), index=["Text 1: AI", "Text 2: Blockchain", "Text 3: Cloud"])
print("Bag of Words representation:")
display(bow_df)

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(), index=["Text 1: AI", "Text 2: Blockchain", "Text 3: Cloud"])
print("TF-IDF scores:")
display(tfidf_df.round(3))

terms = tfidf_vectorizer.get_feature_names_out()
for i, label in enumerate(["Text 1: AI", "Text 2: Blockchain", "Text 3: Cloud"]):
    scores = tfidf_matrix[i].toarray().ravel()
    top_indices = scores.argsort()[::-1][:3]
    top_words = [(terms[j], round(float(scores[j]), 3)) for j in top_indices]
    print(f"{label} top 3 TF-IDF keywords: {top_words}")

### Interpretation

The top terms identify the central topic of each short text: AI-related terms for the first text, ledger and transaction terms for the blockchain text, and storage or resource terms for cloud computing. Because each text is short and the vocabulary differs substantially, words unique to a text tend to receive relatively high TF-IDF scores.

## Q.4 Jaccard and cosine similarity

Two short descriptions of different technologies are preprocessed and compared. Jaccard similarity compares the overlap between unique word sets, while cosine similarity compares the TF-IDF vectors and accounts for term weights.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ai_text = (
    "Artificial intelligence learns patterns from data. "
    "Machine learning models make predictions and support automated decisions. "
    "AI systems can recognize images, understand language, and assist people. "
    "Training data and computing power influence model performance."
)
blockchain_text = (
    "Blockchain records transactions in linked digital blocks. "
    "A distributed network verifies updates without one central controller. "
    "Cryptographic methods help protect the ledger from unauthorized changes. "
    "Smart contracts can automate agreements between participants."
)

def preprocess(text):
    words = re.findall(r"\b[a-z]+\b", text.lower())
    return [word for word in words if word not in stop_words]

ai_tokens = preprocess(ai_text)
blockchain_tokens = preprocess(blockchain_text)
set_ai, set_blockchain = set(ai_tokens), set(blockchain_tokens)

intersection = set_ai & set_blockchain
union = set_ai | set_blockchain
jaccard = len(intersection) / len(union) if union else 0

similarity_vectorizer = TfidfVectorizer()
tfidf_pair = similarity_vectorizer.fit_transform([ai_text, blockchain_text])
cosine = float(cosine_similarity(tfidf_pair[0:1], tfidf_pair[1:2])[0, 0])

print("AI tokens:", ai_tokens)
print("\nBlockchain tokens:", blockchain_tokens)
print("\nShared words:", sorted(intersection))
print(f"Jaccard similarity: {jaccard:.3f}")
print(f"Cosine similarity: {cosine:.3f}")

### Analysis

Jaccard similarity is easy to interpret because it measures the proportion of shared unique words among all unique words in both descriptions. However, it treats every word equally and ignores how often terms occur. TF-IDF cosine similarity considers term weights and is generally more informative when document length or word importance differs. Here, both measures should indicate relatively low similarity because AI and blockchain are distinct technologies, although they may share general technology vocabulary.

## Q.5 Sentiment analysis and word cloud

A set of original product reviews is analyzed with TextBlob. Polarity is between -1 and +1; subjectivity is between 0 and 1. Reviews are classified as positive, negative, or neutral using polarity thresholds. A word cloud is created from the positive reviews.

In [ ]:
# If required in your environment, install once with: %pip install textblob wordcloud
from textblob import TextBlob
from wordcloud import WordCloud

reviews = [
    "The headphones sound excellent and the battery lasts all day.",
    "The phone is fast, reliable, and easy to use.",
    "The delivery was quick and the product quality is fantastic.",
    "The speaker stopped working after two days and customer support was unhelpful.",
    "The app crashes often and the latest update made it worse.",
    "The laptop is too expensive and its battery drains quickly.",
    "The package arrived yesterday.",
    "The product has a screen and two buttons."
]

review_rows = []
for review in reviews:
    sentiment = TextBlob(review).sentiment
    if sentiment.polarity > 0.1:
        label = "Positive"
    elif sentiment.polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"
    review_rows.append({
        "Review": review,
        "Polarity": round(sentiment.polarity, 3),
        "Subjectivity": round(sentiment.subjectivity, 3),
        "Sentiment": label
    })

review_df = pd.DataFrame(review_rows)
display(review_df)

positive_reviews = review_df.loc[review_df["Sentiment"] == "Positive", "Review"].tolist()
positive_text = " ".join(positive_reviews)
if positive_text.strip():
    cloud = WordCloud(width=900, height=450, background_color="white", stopwords=stop_words).generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Positive Reviews")
    plt.show()
else:
    print("No positive reviews were detected, so a word cloud cannot be generated.")

### Interpretation

Positive reviews generally have polarity above 0.1, negative reviews below -0.1, and neutral reviews between these thresholds. Subjectivity estimates whether language expresses opinions rather than objective facts. These values are automated estimates, so sarcasm, context, and mixed reviews may be classified imperfectly. The word cloud highlights frequently occurring words in reviews classified as positive.

## Q.6 Text generation using a neural network

A paragraph of approximately 100 words about responsible technology is used as training text. Keras `Tokenizer` converts words to integer IDs. N-gram sequences are created and used to train a small next-word prediction model. Finally, the model generates three lines from a seed phrase. Since this is a very small training corpus, generated text is illustrative and may not be grammatically perfect.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = (
    "Technology can help people solve problems and share knowledge. "
    "Responsible technology should be useful, reliable, and accessible. "
    "People use computers to learn new skills, create art, communicate with friends, "
    "and explore scientific ideas. Good software protects private information and "
    "supports the needs of its users. Data can help systems recognize patterns, "
    "but people must check results carefully. Creative thinking and teamwork make "
    "technology more valuable. Developers should test their systems, listen to feedback, "
    "and improve their designs. In the future, thoughtful innovation can help communities "
    "learn, work, and solve difficult challenges together."
)

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
input_sequences = []

for line in training_text.lower().split("."):
    sequence = tokenizer.texts_to_sequences([line.strip()])[0]
    for i in range(2, len(sequence) + 1):
        input_sequences.append(sequence[:i])

max_sequence_len = max(len(seq) for seq in input_sequences)
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Vocabulary size:", total_words)
print("Number of training sequences:", len(input_sequences))
print("Input shape:", X.shape, "Target shape:", y.shape)

model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X, y, epochs=150, verbose=0)
print("Final training loss:", round(float(history.history["loss"][-1]), 4))

In [ ]:
def generate_text(seed_text, next_words=18):
    result = seed_text.lower()
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([result])[0]
        sequence = pad_sequences([sequence], maxlen=max_sequence_len - 1, padding="pre")
        predicted_probs = model.predict(sequence, verbose=0)[0]
        predicted_id = int(np.argmax(predicted_probs))
        output_word = next((word for word, index in tokenizer.word_index.items() if index == predicted_id), "")
        if not output_word:
            break
        result += " " + output_word
    return result

seed = "technology can"
print("Seed:", seed)
print("\nGenerated text (3 lines):")
for i in range(3):
    print(f"{i+1}. {generate_text(seed, next_words=12)}")

### Conclusion

This assignment demonstrates common NLP steps: cleaning and tokenizing text, removing stopwords, comparing stems and lemmas, representing documents with counts and TF-IDF, measuring similarity, estimating sentiment, visualizing positive-review vocabulary, and generating text with a small neural language model. Results depend on the selected text, tokenization rules, thresholds, and model training data.